# 🎨 Free Image + Video Generator (easy mode)

1. Click **Runtime → Change runtime type → T4 GPU → Save**.
2. Press the ▶️ button below and wait.
3. Type what you want in the box that appears and press **Make Image** or **Make Video**.

Free Colab gives a few GPU hours per day; when it stops, press play again tomorrow.

In [ ]:
#@title ▶️ Press play. Wait ~10 minutes. A text box and buttons appear below.
import subprocess, sys, os, gc, time
print("1/3 Installing ...")
subprocess.run("pip install -q diffusers transformers accelerate bitsandbytes sentencepiece gradio imageio imageio-ffmpeg protobuf", shell=True, check=True)

import torch, gradio as gr
from diffusers import StableDiffusionXLPipeline, LTXPipeline
from diffusers.utils import export_to_video
from transformers import T5EncoderModel, BitsAndBytesConfig

state = {"name": None, "pipe": None}

def free():
    state["pipe"] = None; state["name"] = None
    gc.collect(); torch.cuda.empty_cache()

def get_image_pipe():
    if state["name"] != "image":
        free()
        print("Loading image model (first time downloads ~7 GB) ...")
        p = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
        p.enable_vae_slicing()
        state["pipe"], state["name"] = p, "image"
    return state["pipe"]

def get_video_pipe():
    if state["name"] != "video":
        free()
        print("Loading video model (first time downloads ~12 GB) ...")
        te = T5EncoderModel.from_pretrained("Lightricks/LTX-Video", subfolder="text_encoder",
                quantization_config=BitsAndBytesConfig(load_in_8bit=True), torch_dtype=torch.float16)
        p = LTXPipeline.from_pretrained("Lightricks/LTX-Video", text_encoder=te, torch_dtype=torch.float16)
        p.enable_model_cpu_offload()
        p.vae.enable_tiling()
        state["pipe"], state["name"] = p, "video"
    return state["pipe"]

NEG = "blurry, low quality, distorted, watermark, text"

def make_image(prompt, count):
    p = get_image_pipe()
    out = p(prompt=prompt, negative_prompt=NEG, num_inference_steps=25, guidance_scale=7,
            width=1024, height=1024, num_images_per_prompt=int(count)).images
    return out

def make_video(prompt, seconds):
    p = get_video_pipe()
    frames = int(seconds) * 24 + 1
    vid = p(prompt=prompt, negative_prompt=NEG, width=768, height=512,
            num_frames=frames, num_inference_steps=30, guidance_scale=3).frames[0]
    path = f"/content/video_{int(time.time())}.mp4"
    export_to_video(vid, path, fps=24)
    return path

print("2/3 Building the app ...")
with gr.Blocks(title="My Free Generator") as app:
    gr.Markdown("# 🎨 My Free Image + Video Generator")
    prompt = gr.Textbox(label="Describe what you want", placeholder="a golden retriever puppy on a beach at sunset, photo, soft light", lines=2)
    with gr.Tab("Image"):
        n = gr.Slider(1, 4, value=1, step=1, label="How many images")
        b1 = gr.Button("Make Image", variant="primary")
        g = gr.Gallery(label="Your images (click one, then the download icon)", columns=2, height=600)
        b1.click(make_image, [prompt, n], g)
    with gr.Tab("Video"):
        s = gr.Slider(2, 5, value=4, step=1, label="Length in seconds")
        b2 = gr.Button("Make Video (takes a few minutes)", variant="primary")
        v = gr.Video(label="Your video")
        b2.click(make_video, [prompt, s], v)
    gr.Markdown("First image and first video each take extra minutes to download the model. After that, images take ~30 s and videos 2-5 min.")

print("3/3 Ready! Use the app below. (If it does not show, open the backup link printed under it.)")
app.queue().launch(share=True, inline=True, debug=False)
